In [1]:
import numpy as np

def check_decomposition(U, V, W, m, n, p):
    """U: (R, m*n)  V: (R, n*p)  W: (R, m*p)
    验证 sum_r (u_r ⊗ v_r ⊗ w_r) 是否等于矩阵乘法张量 T_{m,n,p}。"""
    R = U.shape[0]
    T = np.zeros((m * n, n * p, m * p))
    for i in range(m):
        for j in range(n):
            for k in range(p):
                T[i * n + j, j * p + k, i * p + k] = 1.0
    That = np.einsum('ra,rb,rc->abc', U, V, W)   # 秩 R 分解重建的张量
    return np.allclose(T, That, atol=1e-9), R

def matmul_via_decomposition(A, B, U, V, W, m, n, p):
    """用分解给出的算法计算 C = A @ B，返回 (C, 标量乘法次数)。"""
    a = A.reshape(-1); b = B.reshape(-1)
    C = np.zeros(m * p)
    for r in range(U.shape[0]):
        C += (U[r] @ a) * (V[r] @ b) * W[r]      # 一次标量乘法 + 一次秩一累加
    return C.reshape(m, p), U.shape[0]

In [2]:
import numpy as np

# ---------- 硬约束：设计必须合法 ----------
def feasible(idxs, N, m):
    """下标必须互异、数量正确、且在候选集范围内。"""
    a = np.asarray(idxs)
    if a.shape[0] != N:            return False
    if a.min() < 0 or a.max() >= m: return False
    return len(np.unique(a)) == N

# ---------- 打分：log det 信息矩阵 ----------
def d_criterion(idxs, F, ridge=1e-9):
    """F: (m, d) 候选点的回归向量；返回 log det(X_S^T X_S)。"""
    X = F[np.asarray(idxs)]
    M = X.T @ X + ridge * np.eye(X.shape[1])
    sign, logdet = np.linalg.slogdet(M)
    return logdet if sign > 0 else -np.inf

# ---------- 评估器：训练实例与留出实例分离 ----------
def evaluate(program, F_train, F_holdout, N):
    """program: 被搜索出的构造规则；返回 (训练分, 留出分)。
    留出分只在最终报告时调用一次，绝不回传给搜索过程。"""
    s_tr, s_ho = [], []
    for F in (F_train, F_holdout):
        idxs = program(F, N)                        # 被搜索的规则在给定候选集上构造设计
        if not feasible(idxs, N, F.shape[0]):
            return -np.inf, -np.inf
        (s_tr if F is F_train else s_ho).append(d_criterion(idxs, F))
    return float(np.mean(s_tr)), float(np.mean(s_ho))

# ---------- 单元测试：评估器本身必须正确 ----------
def _test_evaluator():
    rng = np.random.default_rng(0)
    d, N = 3, 6
    # 退化情形：d 个正交方向时，log det 应当等于 0（单位信息矩阵）
    F = np.eye(d)
    assert np.isclose(d_criterion(np.arange(d), F, ridge=0.0), 0.0)
    # 单调性：把某个点换成范数更大的点，D 准则不应下降
    x, y = np.array([1.0, 0, 0]), np.array([2.0, 0, 0])
    F2 = np.vstack([np.eye(d), x]); F3 = np.vstack([np.eye(d), y])
    assert d_criterion(np.arange(d + 1), F3) >= d_criterion(np.arange(d + 1), F2) - 1e-12
    # 已知最优：全候选集上任取 N 个，不得优于暴力枚举的最优（小规模才可行）
    from itertools import combinations
    m = 8; F4 = rng.normal(size=(m, d))
    brute = max(d_criterion(np.array(c), F4) for c in combinations(range(m), min(N, m)))
    assert d_criterion(np.arange(min(N, m)), F4) <= brute + 1e-12

_test_evaluator()

In [3]:
import numpy as np

def solve(F, N, priority, eps=1e-6):
    """F: (m, d) 候选回归向量；N: 设计点数；priority: 被进化的决策规则。
    返回选中的下标数组。"""
    m, d = F.shape
    M = eps * np.eye(d)
    chosen, used = [], np.zeros(m, dtype=bool)
    for _ in range(N):
        scores = np.full(m, -np.inf)
        for j in range(m):
            if not used[j]:
                scores[j] = priority(F[j], M, F)
        j = int(np.nanargmax(scores))
        chosen.append(j); used[j] = True
        x = F[j]
        # 秩一更新；若要加速可维护 M^{-1} 并用 Sherman–Morrison 更新
        M = M + np.outer(x, x)
    return np.array(chosen)

# 待进化的函数的初始（平凡）版本：退化为标准贪心 D-最优的一步准则
def priority_trivial(x, M, F):
    Minv = np.linalg.pinv(M)
    return float(x @ Minv @ x)

# 经典基线：坐标交换（Coordinate Exchange, Meyer & Nachtsheim 1995），用于对照
def coordinate_exchange(F, N, rng, sweeps=20):
    m, d = F.shape
    idx = rng.choice(m, size=N, replace=False)
    def crit(S):
        X = F[S]; M = X.T @ X + 1e-9 * np.eye(d)
        s, ld = np.linalg.slogdet(M)
        return ld if s > 0 else -np.inf
    best = crit(idx)
    for _ in range(sweeps):
        improved = False
        for t in range(N):
            for j in range(m):
                if j in idx: continue
                trial = idx.copy(); trial[t] = j
                c = crit(trial)
                if c > best + 1e-12:
                    idx, best, improved = trial, c, True
        if not improved: break
    return idx, best

In [ ]:
import sys
sys.stdout.reconfigure(encoding="utf-8")
import math
import random
from collections import Counter
from itertools import combinations

K = 6          # 刻度数（固定）
HORIZON = 40   # 构造器扫描的位置上限
TOP_M = 3      # 每步只在优先级前 3 名的可行位置中抽样
RECORD = 17    # 已知最优 Golomb 尺长度（文献值）

# ---------- 合法性检查：两版实现 ----------
def diffs_of(S):
    return [b - a for a, b in combinations(sorted(S), 2)]

def ok_v1(d):
    """第一版写法：枚举差值后只比较相邻两项——忘了先排序。"""
    return all(x != y for x, y in zip(d, d[1:]))

def ok_v2(d):
    """修正版：把全部差值放进集合，按个数核对。"""
    return len(d) == len(set(d))

def make_checker(ok):
    """构造器与评估器共用同一套检查——这正是第一版埋下的隐患。"""
    def inc(S, x):
        return ok(diffs_of(S + [x]))
    def final(S):
        return ok(diffs_of(S))
    return inc, final

# ---------- 被进化的"程序"：带温度的贪心构造器 ----------
def build(w, ok, rng):
    """w = (a, b, c, lg_t)。优先级 = a*x + b*最近间距 + c；每步在优先级
    前 TOP_M 的可行位置中按 softmax（温度 exp(lg_t)）抽一个。"""
    inc, final = make_checker(ok)
    a, b, c, lg_t = w
    S = [0]
    while len(S) < K:
        cands = [x for x in range(S[-1] + 1, HORIZON) if inc(S, x)]
        if not cands:
            return None
        sc = {x: a * x + b * min(x - s for s in S) + c for x in cands}
        top = sorted(cands, key=lambda x: -sc[x])[:TOP_M]
        m = max(sc[x] for x in top)
        wts = [math.exp((sc[x] - m) / math.exp(lg_t)) for x in top]
        S.append(rng.choices(top, weights=wts)[0])
    return tuple(S) if final(S) else None

def evaluate(w, ok, seed, n_rep=4):
    """评估协议：同一随机流连续构造 n_rep 次，取最短合法长度（记负分）。"""
    rng = random.Random(seed)
    lens = [build(w, ok, rng) for _ in range(n_rep)]
    lens = [s[-1] for s in lens if s]
    return -min(lens) if lens else -1e9

def best_build(w, ok, seed, n_rep=4):
    """按与 evaluate 完全相同的协议复现冠军的构造。"""
    rng = random.Random(seed)
    best = None
    for _ in range(n_rep):
        s = build(w, ok, rng)
        if s and (best is None or s[-1] < best[-1]):
            best = s
    return best

# ---------- 进化循环 ----------
def mutate(w, rng):
    v = [wi + rng.gauss(0.0, 1.0) for wi in w]
    if rng.random() < 0.25:
        v[rng.randrange(4)] = 0.0          # "简化"算子：随机置零一个权重
    return v

def run_search(ok, seed, tag, gens=25, pop=12, lam=8):
    rng = random.Random(seed)
    pool = [(-1.0, 0.0, 0.0, -3.0)]        # 初始程序：近确定性的贪心
    scores = [evaluate(w, ok, seed) for w in pool]
    for g in range(1, gens + 1):
        for _ in range(lam):
            parent = pool[rng.randrange(len(pool) // 2 + 1)]
            child = mutate(parent, rng)
            pool.append(child)
            scores.append(evaluate(child, ok, seed))
        order = sorted(range(len(pool)), key=lambda i: -scores[i])[:pop]
        pool = [pool[i] for i in order]
        scores = [scores[i] for i in order]
        print(f"[{tag}] 第 {g:2d} 代  最佳长度 {-max(scores):2d}")
    i = scores.index(max(scores))
    return pool[i]

def audit(S):
    """独立复核：换一种写法，用计数器核对差值多重集。"""
    cnt = Counter(diffs_of(S))
    bad = {d: n for d, n in cnt.items() if n > 1}
    return (not bad), bad

if __name__ == "__main__":
    print("=== 第一轮：带 bug 的评估器（ok_v1）===")
    w1 = run_search(ok_v1, seed=7, tag="v1")
    champ1 = best_build(w1, ok_v1, seed=7)
    valid, bad = audit(champ1)
    print(f"[v1] 冠军长度 {champ1[-1]}，刻度 {list(champ1)}")
    print(f"[v1] 独立复核：{'合法' if valid else f'非法，重复差值 {bad}'}")
    print(f"[v1] 已知最优 G({K}) = {RECORD}，冠军 {champ1[-1]} < {RECORD} -> 红旗：先怀疑评估器")
    print()
    print("=== 第二轮：修正后的评估器（ok_v2），三个种子 ===")
    for sd in (1, 2, 3):
        w2 = run_search(ok_v2, seed=sd, tag=f"v2-s{sd}")
        champ2 = best_build(w2, ok_v2, seed=sd)
        valid, bad = audit(champ2)
        print(f"[v2-s{sd}] 冠军长度 {champ2[-1]}，刻度 {list(champ2)}，独立复核：{'合法' if valid else '非法'}")
    print()
    print("=== 冠军复评：换 20 个新种子、每个种子按评估协议跑 4 次取最优 ===")
    lens = sorted(c[-1] for s2 in range(1000, 1020) if (c := best_build(w2, ok_v2, seed=s2)))
    hit = sum(1 for L in lens if L == RECORD)
    print(f"20 个种子中达到已知最优 {RECORD} 的有 {hit} 个；"
          f"长度中位数 {lens[10]}，最差 {max(lens)}，全部合法")

In [ ]:
import sys
sys.stdout.reconfigure(encoding="utf-8")
import math
import random
from collections import Counter
from itertools import combinations

K, HORIZON, TOP_M = 6, 200, 3
RECORD = 17
LAM = 0.4    # 代理指标给每个"重复差值"的罚分（硬约束版本等价于罚分为无穷大）

def diffs_of(S):
    return [b - a for a, b in combinations(sorted(S), 2)]

def violations(S):
    """重复差值超出次数之和：完全合法为 0。"""
    cnt = Counter(diffs_of(S))
    return sum(n - 1 for n in cnt.values() if n > 1)

def build(w, rng):
    """与第一个脚本相同的构造器，只是增量检查恒通过（合法性全部交给评估器）。"""
    a, b, c, lg_t = w
    S = [0]
    while len(S) < K:
        cands = [x for x in range(S[-1] + 1, HORIZON)]
        if not cands:
            return None
        sc = {x: a * x + b * min(x - s for s in S) + c for x in cands}
        top = sorted(cands, key=lambda x: -sc[x])[:TOP_M]
        m = max(sc[x] for x in top)
        wts = [math.exp((sc[x] - m) / math.exp(lg_t)) for x in top]
        S.append(rng.choices(top, weights=wts)[0])
    return tuple(S)

def proxy_score(S):
    """平滑代理指标：越短越好，重复差值只罚 0.4 分而不否决。"""
    return -S[-1] + LAM * violations(S)

def evaluate(w, seed, n_rep=4):
    """评估协议与第一个脚本一致：同一随机流构造 4 次，取代理分最高的一次。"""
    rng = random.Random(seed)
    ps = [proxy_score(s) for s in (build(w, rng) for _ in range(n_rep)) if s]
    return max(ps) if ps else -1e9

def true_report(w, seed, n_rep=4):
    """把同一程序交给硬评估器：报告 4 次构造中最好的真实成绩。"""
    rng = random.Random(seed)
    best = None
    for _ in range(n_rep):
        s = build(w, rng)
        if s and violations(s) == 0 and (best is None or s[-1] < best[-1]):
            best = s
    return best

def mutate(w, rng):
    v = [wi + rng.gauss(0.0, 1.0) for wi in w]
    if rng.random() < 0.25:
        v[rng.randrange(4)] = 0.0
    return v

if __name__ == "__main__":
    seed = 7
    rng = random.Random(seed)
    pool = [(-1.0, 0.0, 0.0, -3.0)]
    scores = [evaluate(w, seed) for w in pool]
    for g in range(1, 21):                 # 20 代，种群 12，每代新增 8
        for _ in range(8):
            child = mutate(pool[rng.randrange(len(pool) // 2 + 1)], rng)
            pool.append(child)
            scores.append(evaluate(child, seed))
        order = sorted(range(len(pool)), key=lambda i: -scores[i])[:12]
        pool = [pool[i] for i in order]
        scores = [scores[i] for i in order]
        champ = true_report(pool[0], seed)
        rep = (f"长度 {champ[-1]}，合法" if champ else "4 次构造全部非法")
        print(f"第 {g:2d} 代  代理最佳分 {max(scores):7.2f}   冠军真实成绩：{rep}")
    S = build(pool[0], random.Random(0)) or (0, 1, 2, 3, 4, 5)
    bad = {d: n for d, n in Counter(diffs_of(S)).items() if n > 1}
    print()
    print(f"最终冠军（按代理分选出）：刻度 {list(S)}")
    print(f"  长度 {S[-1]}，重复差值 {bad}")
    print(f"  代理分 {proxy_score(S):.2f}（高于合法最优 {RECORD} 的 {-RECORD:.1f}）")
    print(f"  硬评估器判定：{'合法' if not bad else '非法'}")